In [5]:
from pathlib import Path
import subprocess
import sys

project_root = Path.cwd()
if not (project_root / "tests" / "test_flask_app.py").exists():
    project_root = project_root.parent

test_file = project_root / "tests" / "test_flask_app.py"
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-v", "-rA", str(test_file)],
    cwd=str(project_root),
    check=False,
    text=True,
    capture_output=False,
 )
print(f"pytest exit code: {result.returncode}")

============================= test session starts ==============================
platform darwin -- Python 3.13.12, pytest-9.0.2, pluggy-1.6.0 -- /opt/homebrew/opt/python@3.13/bin/python3.13
cachedir: .pytest_cache
rootdir: /Users/alex/Desktop/COMP30830-David
collecting ... collected 5 items

tests/test_flask_app.py::test_weather_success PASSED                     [ 20%]
tests/test_flask_app.py::test_weather_api_unavailable PASSED             [ 40%]
tests/test_flask_app.py::test_stations_success PASSED                    [ 60%]
tests/test_flask_app.py::test_stations_handles_exception PASSED          [ 80%]
tests/test_flask_app.py::test_station_sql_info_not_found PASSED          [100%]

==================================== PASSES ====================================
=========================== short test summary info ============================
PASSED tests/test_flask_app.py::test_weather_success
PASSED tests/test_flask_app.py::test_weather_api_unavailable
PASSED tests/test_flask_app.p

In [4]:
import sys
from pathlib import Path

import pytest

# Notebook 环境没有 __file__，用当前工作目录推断项目根目录
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "flaskapi").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

FLASKAPI_DIR = PROJECT_ROOT / "flaskapi"
if str(FLASKAPI_DIR) not in sys.path:
    sys.path.insert(0, str(FLASKAPI_DIR))

import app as app_module


@pytest.fixture()
def client():
    app_module.app.config["TESTING"] = True
    with app_module.app.test_client() as test_client:
        yield test_client


def test_weather_success(client, monkeypatch):
    monkeypatch.setattr(app_module, "get_weather", lambda: {"temp": 12, "desc": "cloudy"})

    resp = client.get("/weather")

    assert resp.status_code == 200
    assert resp.get_json() == {"temp": 12, "desc": "cloudy"}


def test_weather_api_unavailable(client, monkeypatch):
    monkeypatch.setattr(app_module, "get_weather", lambda: None)

    resp = client.get("/weather")

    assert resp.status_code == 500
    assert "error" in resp.get_json()


def test_stations_success(client, monkeypatch):
    fake_data = [{"number": 42, "name": "Test Station"}]
    monkeypatch.setattr(app_module, "get_latest_stations_view", lambda: fake_data)

    resp = client.get("/stations")

    assert resp.status_code == 200
    assert resp.get_json() == fake_data


def test_stations_handles_exception(client, monkeypatch):
    def raise_error():
        raise RuntimeError("db down")

    monkeypatch.setattr(app_module, "get_latest_stations_view", raise_error)

    resp = client.get("/stations")

    assert resp.status_code == 500
    assert "error" in resp.get_json()


def test_station_sql_info_not_found(client, monkeypatch):
    monkeypatch.setattr(app_module, "get_station_sql", lambda station_id: None)

    resp = client.get("/stations_SQL/99999/info")

    assert resp.status_code == 404
    assert resp.get_json()["error"] == "Station not found"

print("app.py unit tests loaded")

app.py unit tests loaded


In [7]:
from pathlib import Path
import subprocess
import sys
import textwrap

project_root = Path.cwd()
if not (project_root / "bikeinfo").exists():
    project_root = project_root.parent

test_file = project_root / "tests" / "test_bikeinfo_cells.py"
test_file.write_text(
    textwrap.dedent(
        '''
        import ast
        import json
        from datetime import datetime, timezone
        from pathlib import Path
        from types import SimpleNamespace


        def _load_function_from_script(script_path: Path, func_name: str, injected_globals: dict):
            source = script_path.read_text(encoding="utf-8")
            module_ast = ast.parse(source, filename=str(script_path))
            for node in module_ast.body:
                if isinstance(node, ast.FunctionDef) and node.name == func_name:
                    target = node
                    break
            else:
                raise AssertionError(f"Function {func_name} not found in {script_path}")

            func_module = ast.Module(body=[target], type_ignores=[])
            compiled = compile(func_module, filename=str(script_path), mode="exec")
            ns = dict(injected_globals)
            exec(compiled, ns)
            return ns[func_name]


        def test_import_once_from_cell04_transforms_and_inserts(monkeypatch):
            script = Path("bikeinfo/bikeapi_cells/cell04_import_api_to_database.py")

            class FakeResponse:
                def raise_for_status(self):
                    return None

                def json(self):
                    return [
                        {
                            "number": 100,
                            "contract_name": "dublin",
                            "name": "A",
                            "address": "addr",
                            "position": {"lat": 53.0, "lng": -6.0},
                            "banking": 1,
                            "bonus": 0,
                            "bike_stands": 20,
                            "available_bike_stands": 7,
                            "available_bikes": 13,
                            "status": "OPEN",
                            "last_update": 1700000000000,
                        },
                        {
                            "number": None,
                            "name": "skip",
                        },
                    ]

            class FakeRequests:
                @staticmethod
                def get(url, timeout):
                    assert timeout == 20
                    return FakeResponse()

            calls = []

            class FakeConn:
                def execute(self, stmt, params):
                    calls.append((stmt, params))

            class FakeBegin:
                def __enter__(self):
                    return FakeConn()

                def __exit__(self, exc_type, exc, tb):
                    return False

            class FakeEngine:
                def begin(self):
                    return FakeBegin()

            import_once = _load_function_from_script(
                script,
                "import_once",
                {
                    "requests": FakeRequests,
                    "config": SimpleNamespace(BIKE_STATUS_URL="http://example.test"),
                    "datetime": datetime,
                    "timezone": timezone,
                    "engine": FakeEngine(),
                    "station_insert": "station_stmt",
                    "availability_insert": "availability_stmt",
                },
            )

            import_once()

            assert len(calls) == 2
            station_stmt, station_params = calls[0]
            availability_stmt, availability_params = calls[1]
            assert station_stmt == "station_stmt"
            assert availability_stmt == "availability_stmt"
            assert len(station_params) == 1
            assert station_params[0]["number"] == 100
            assert len(availability_params) == 1
            assert availability_params[0]["available_bikes"] == 13
            assert availability_params[0]["last_update"] is not None


        def test_fetch_and_save_once_from_cell01_writes_json(tmp_path):
            script = Path("bikeinfo/bikeapi_cells/cell01_fetch_status_to_json.py")
            payload = [{"number": 1, "name": "X"}]

            class FakeResponse:
                status_code = 200

                def raise_for_status(self):
                    return None

                def json(self):
                    return payload

            class FakeRequests:
                @staticmethod
                def get(url, timeout):
                    assert timeout == 20
                    return FakeResponse()

            fetch_and_save_once = _load_function_from_script(
                script,
                "fetch_and_save_once",
                {
                    "datetime": datetime,
                    "timezone": timezone,
                    "json": json,
                    "requests": FakeRequests,
                    "config": SimpleNamespace(BIKE_STATUS_URL="http://example.test"),
                    "output_dir": tmp_path,
                },
            )

            fetch_and_save_once()

            files = list(tmp_path.glob("station_status_*.json"))
            assert len(files) == 1
            saved = json.loads(files[0].read_text(encoding="utf-8"))
            assert saved == payload
        '''
    ),
    encoding="utf-8",
)

print(f"created: {test_file}")

result = subprocess.run(
    [sys.executable, "-m", "pytest", "-v", "-rA", str(test_file)],
    cwd=str(project_root),
    check=False,
    text=True,
    capture_output=False,
)
print(f"pytest exit code: {result.returncode}")

created: /Users/alex/Desktop/COMP30830-David/tests/test_bikeinfo_cells.py
============================= test session starts ==============================
platform darwin -- Python 3.13.12, pytest-9.0.2, pluggy-1.6.0 -- /opt/homebrew/opt/python@3.13/bin/python3.13
cachedir: .pytest_cache
rootdir: /Users/alex/Desktop/COMP30830-David
collecting ... collected 2 items

tests/test_bikeinfo_cells.py::test_import_once_from_cell04_transforms_and_inserts PASSED [ 50%]
tests/test_bikeinfo_cells.py::test_fetch_and_save_once_from_cell01_writes_json PASSED [100%]

==================================== PASSES ====================================
_____________ test_import_once_from_cell04_transforms_and_inserts ______________
----------------------------- Captured stdout call -----------------------------
API import completed: stations=1, availability=1
_______________ test_fetch_and_save_once_from_cell01_writes_json _______________
----------------------------- Captured stdout call ------------------